# SpotWhisperer Evaluation Results Aggregation

Aggregates results from all three benchmarks (HEST, Lung, CellWhisperer) across all dataset combinations and seeds.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import json
import matplotlib

In [ ]:
print("Aggregating SpotWhisperer evaluation results...")
print(f"Input files:")
print(f"  HEST results: {len(snakemake.input.hest_results)} files")
print(f"  Lung results: {len(snakemake.input.lung_results)} files")
print(f"  CellWhisperer results: {len(snakemake.input.cellwhisperer_results)} files")

In [ ]:
def extract_metadata_from_path(file_path):
    """Extract dataset_combo and seed from file path"""
    parts = Path(file_path).parts
    
    # Find indices of key parts
    benchmark_idx = None
    for i, part in enumerate(parts):
        if part in ['hest', 'lung', 'cellwhisperer']:
            benchmark_idx = i
            break
    
    if benchmark_idx is None:
        return None, None, None
    
    benchmark_type = parts[benchmark_idx]
    
    if benchmark_idx + 1 < len(parts):
        dataset_combo = parts[benchmark_idx + 1]
    else:
        return benchmark_type, None, None
    
    if benchmark_idx + 2 < len(parts):
        seed_part = parts[benchmark_idx + 2]
        if seed_part.startswith('seed_'):
            seed = seed_part.replace('seed_', '')
        else:
            seed = 'unknown'
    else:
        seed = 'unknown'
    
    return benchmark_type, dataset_combo, seed

In [ ]:
# Aggregate all results
all_results = []

# Process HEST results
for hest_file in snakemake.input.hest_results:
    benchmark_type, dataset_combo, seed = extract_metadata_from_path(hest_file)
    
    if Path(hest_file).exists():
        try:
            with open(hest_file, 'r') as f:
                data = json.load(f)
            
            result_row = {
                'benchmark': 'HEST',
                'dataset_combo': dataset_combo,
                'seed': seed,
                'file_path': hest_file
            }
            result_row.update(data)
            all_results.append(result_row)
            print(f"Loaded HEST results: {dataset_combo}/seed_{seed}")
        except Exception as e:
            print(f"Error loading HEST file {hest_file}: {e}")
    else:
        print(f"HEST file not found: {hest_file}")

# Process Lung results
for lung_file in snakemake.input.lung_results:
    benchmark_type, dataset_combo, seed = extract_metadata_from_path(lung_file)
    
    if Path(lung_file).exists():
        try:
            df = pd.read_csv(lung_file)
            
            # Convert DataFrame to dict, taking first row if multiple
            data = df.iloc[0].to_dict() if len(df) > 0 else {}
            
            result_row = {
                'benchmark': 'Lung',
                'dataset_combo': dataset_combo,
                'seed': seed,
                'file_path': lung_file
            }
            result_row.update(data)
            all_results.append(result_row)
            print(f"Loaded Lung results: {dataset_combo}/seed_{seed}")
        except Exception as e:
            print(f"Error loading Lung file {lung_file}: {e}")
    else:
        print(f"Lung file not found: {lung_file}")

# Process CellWhisperer results  
for cw_file in snakemake.input.cellwhisperer_results:
    benchmark_type, dataset_combo, seed = extract_metadata_from_path(cw_file)
    
    if Path(cw_file).exists():
        try:
            df = pd.read_csv(cw_file)
            
            # Convert DataFrame to dict, taking first row if multiple
            data = df.iloc[0].to_dict() if len(df) > 0 else {}
            
            result_row = {
                'benchmark': 'CellWhisperer',
                'dataset_combo': dataset_combo,
                'seed': seed,
                'file_path': cw_file
            }
            result_row.update(data)
            all_results.append(result_row)
            print(f"Loaded CellWhisperer results: {dataset_combo}/seed_{seed}")
        except Exception as e:
            print(f"Error loading CellWhisperer file {cw_file}: {e}")
    else:
        print(f"CellWhisperer file not found: {cw_file}")

print(f"\nTotal results loaded: {len(all_results)}")

In [ ]:
# Create combined DataFrame
if all_results:
    combined_df = pd.DataFrame(all_results)
    print(f"Combined results shape: {combined_df.shape}")
    print(f"Benchmarks: {combined_df['benchmark'].unique()}")
    print(f"Dataset combinations: {combined_df['dataset_combo'].unique()}")
    print(f"Seeds: {combined_df['seed'].unique()}")
else:
    combined_df = pd.DataFrame()
    print("No results to combine.")

In [ ]:
# Save aggregated results
Path(snakemake.output.aggregated_results).parent.mkdir(parents=True, exist_ok=True)

if not combined_df.empty:
    combined_df.to_csv(snakemake.output.aggregated_results, index=False)
    print(f"Aggregated results saved to: {snakemake.output.aggregated_results}")
else:
    # Create empty file to satisfy Snakemake
    pd.DataFrame().to_csv(snakemake.output.aggregated_results, index=False)
    print("Saved empty aggregated results file.")

In [ ]:
# Create summary visualization
if not combined_df.empty and len(combined_df) > 0:
    # Create a comprehensive summary plot
    fig, axes = plt.subplots(2, 2, figsize=(15, 12))
    fig.suptitle('SpotWhisperer Evaluation Summary', fontsize=16)
    
    # 1. Performance by benchmark type
    ax1 = axes[0, 0]
    if 'accuracy' in combined_df.columns:
        benchmark_perf = combined_df.groupby('benchmark')['accuracy'].agg(['mean', 'std']).reset_index()
        bars1 = ax1.bar(benchmark_perf['benchmark'], benchmark_perf['mean'], 
                       yerr=benchmark_perf['std'], capsize=5)
        ax1.set_title('Average Accuracy by Benchmark')
        ax1.set_ylabel('Accuracy')
        ax1.set_ylim(0, 1)
        
        for bar, mean_val in zip(bars1, benchmark_perf['mean']):
            ax1.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.02,
                   f'{mean_val:.3f}', ha='center', va='bottom')
    else:
        ax1.text(0.5, 0.5, 'Accuracy data not available', 
               transform=ax1.transAxes, ha='center', va='center')
    
    # 2. Performance by dataset combination
    ax2 = axes[0, 1]
    if 'accuracy' in combined_df.columns and len(combined_df['dataset_combo'].unique()) > 1:
        combo_perf = combined_df.groupby('dataset_combo')['accuracy'].agg(['mean', 'std']).reset_index()
        bars2 = ax2.bar(range(len(combo_perf)), combo_perf['mean'], 
                       yerr=combo_perf['std'], capsize=5)
        ax2.set_title('Average Accuracy by Dataset Combination')
        ax2.set_ylabel('Accuracy')
        ax2.set_xticks(range(len(combo_perf)))
        ax2.set_xticklabels(combo_perf['dataset_combo'], rotation=45, ha='right')
        ax2.set_ylim(0, 1)
        
        for bar, mean_val in zip(bars2, combo_perf['mean']):
            ax2.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.02,
                   f'{mean_val:.3f}', ha='center', va='bottom')
    else:
        ax2.text(0.5, 0.5, 'Dataset combo data not available', 
               transform=ax2.transAxes, ha='center', va='center')
    
    # 3. Variability across seeds
    ax3 = axes[1, 0]
    if 'accuracy' in combined_df.columns and len(combined_df['seed'].unique()) > 1:
        seed_perf = combined_df.groupby('seed')['accuracy'].agg(['mean', 'std']).reset_index()
        bars3 = ax3.bar(seed_perf['seed'], seed_perf['mean'], 
                       yerr=seed_perf['std'], capsize=5)
        ax3.set_title('Average Accuracy by Seed')
        ax3.set_ylabel('Accuracy')
        ax3.set_xlabel('Seed')
        ax3.set_ylim(0, 1)
        
        for bar, mean_val in zip(bars3, seed_perf['mean']):
            ax3.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.02,
                   f'{mean_val:.3f}', ha='center', va='bottom')
    else:
        ax3.text(0.5, 0.5, 'Seed variability data not available', 
               transform=ax3.transAxes, ha='center', va='center')
    
    # 4. Heatmap of performance by dataset combo and benchmark
    ax4 = axes[1, 1]
    if 'accuracy' in combined_df.columns and len(combined_df) > 0:
        pivot_data = combined_df.pivot_table(values='accuracy', 
                                           index='dataset_combo', 
                                           columns='benchmark', 
                                           aggfunc='mean')
        if not pivot_data.empty:
            sns.heatmap(pivot_data, annot=True, fmt='.3f', cmap='Blues', 
                       ax=ax4, cbar_kws={'label': 'Accuracy'})
            ax4.set_title('Performance Heatmap')
            ax4.set_xlabel('Benchmark')
            ax4.set_ylabel('Dataset Combination')
        else:
            ax4.text(0.5, 0.5, 'Insufficient data for heatmap', 
                   transform=ax4.transAxes, ha='center', va='center')
    else:
        ax4.text(0.5, 0.5, 'Heatmap data not available', 
               transform=ax4.transAxes, ha='center', va='center')
    
    plt.tight_layout()
    plt.savefig(snakemake.output.summary_plot, dpi=300, bbox_inches='tight')
    print(f"Summary plot saved to: {snakemake.output.summary_plot}")
    
else:
    # Create empty plot
    fig, ax = plt.subplots(figsize=(10, 6))
    ax.text(0.5, 0.5, 'No evaluation results available to plot', 
           transform=ax.transAxes, ha='center', va='center', fontsize=14)
    ax.set_title('SpotWhisperer Evaluation Summary')
    plt.savefig(snakemake.output.summary_plot, dpi=300, bbox_inches='tight')
    print(f"Empty summary plot saved to: {snakemake.output.summary_plot}")

plt.close('all')

In [ ]:
# Print summary statistics
if not combined_df.empty:
    print("\n=== SpotWhisperer Evaluation Summary ===")
    print(f"Total evaluations: {len(combined_df)}")
    print(f"Benchmarks: {', '.join(combined_df['benchmark'].unique())}")
    print(f"Dataset combinations: {', '.join(combined_df['dataset_combo'].unique())}")
    print(f"Seeds: {', '.join(combined_df['seed'].unique())}")
    
    if 'accuracy' in combined_df.columns:
        print(f"\nOverall performance:")
        print(f"  Mean accuracy: {combined_df['accuracy'].mean():.3f} ± {combined_df['accuracy'].std():.3f}")
        print(f"  Best accuracy: {combined_df['accuracy'].max():.3f}")
        print(f"  Worst accuracy: {combined_df['accuracy'].min():.3f}")
        
        # Best performing combination
        best_idx = combined_df['accuracy'].idxmax()
        best_row = combined_df.iloc[best_idx]
        print(f"\nBest performing combination:")
        print(f"  Dataset: {best_row['dataset_combo']}")
        print(f"  Benchmark: {best_row['benchmark']}")
        print(f"  Seed: {best_row['seed']}")
        print(f"  Accuracy: {best_row['accuracy']:.3f}")
    
    print("=" * 40)
else:
    print("No evaluation results to summarize.")